# 🚗 Fotoperitación — Entrenamiento en GPU (Colab)

Entrena el modelo de **daños** (YOLOv11m-seg, 2 fases) en GPU — la ruta prevista en el plan (MPS local es demasiado lento).

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno → Acelerador: GPU` (T4 gratis, o A100/L4 en Pro).

Flujo: GPU → código → datos → (sanity) → entrenamiento → evaluar → guardar en Drive.

In [ ]:
!nvidia-smi

## 1. Montar Google Drive
Para leer código/datos y guardar el modelo entrenado.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Carpeta de trabajo en tu Drive (crea ahí y sube los .zip):
WORK = '/content/drive/MyDrive/fotoperitacion'
!mkdir -p "$WORK"
print("WORK =", WORK)

## 2. Código del proyecto
Elige **A** (GitHub) o **B** (zip en Drive).

In [ ]:
# --- Opción A: clonar desde GitHub (si lo has subido) ---
# !git clone https://github.com/<TU_USUARIO>/Comp_vision.git /content/Comp_vision

# --- Opción B: descomprimir code.zip subido a Drive ---
!mkdir -p /content/Comp_vision
!unzip -q -o "$WORK/code.zip" -d /content/Comp_vision

%cd /content/Comp_vision
!ls scripts configs

In [ ]:
!pip -q install ultralytics   # trae torch+CUDA, opencv, pyyaml, etc.

## 3. Datos
Elige **A** (re-ingesta en Colab, sin subir 3 GB) o **B** (subir `data_final.zip`).

In [ ]:
# --- Opción A (recomendada): regenerar el dataset en Colab ---
# Necesita kaggle.json (Kaggle → Settings → API → Create New Token)
!pip -q install kaggle huggingface_hub
from google.colab import files
files.upload()          # selecciona tu kaggle.json
!mkdir -p /root/.kaggle && mv kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
!python scripts/download_datasets.py --datasets vehide,cardd
!python scripts/unify_to_yolo.py     # genera data/final + configs/dataset.yaml con rutas correctas

In [ ]:
# --- Opción B: usar el data/final ya preparado (subido como data_final.zip) ---
# !unzip -q -o "$WORK/data_final.zip" -d /content/Comp_vision
# import yaml, pathlib
# p = pathlib.Path('configs/dataset.yaml')
# d = yaml.safe_load(p.read_text()); d['path'] = '/content/Comp_vision/data/final'
# p.write_text(yaml.safe_dump(d, sort_keys=False, allow_unicode=True)); print(p.read_text())

## 4. Sanity check (opcional)
3 epochs cortos para confirmar que todo corre en GPU antes del run largo.

In [ ]:
!python scripts/train.py --epochs 3 --imgsz 640 --batch 16 --phase1-only

## 5. Entrenamiento completo (2 fases)
Fase 1 (backbone congelado, 20 ep) → Fase 2 (fine-tuning, 280 ep) @ 1024px.

⏱️ En **T4 gratis** esto puede superar el límite de sesión (~12 h). Opciones: usar **A100/L4** (Pro), reducir `--epochs-phase2` (p.ej. 100), o bajar `--imgsz`. Ultralytics guarda checkpoints cada época.

In [ ]:
# T4 (16 GB): si da OOM, baja a --batch 8
!python scripts/train.py --batch 16 --imgsz 1024
# Variante más corta para T4 gratis:
# !python scripts/train.py --batch 16 --imgsz 1024 --epochs-phase2 100

## 6. Evaluar y guardar en Drive

In [ ]:
!python scripts/evaluate.py --model runs/damage_seg/phase2_finetune/weights/best.pt
# Copiar modelo + métricas + curvas a Drive:
!cp -r runs/damage_seg "$WORK/"
!cp -r evaluation_results "$WORK/" 2>/dev/null || true
print("✅ Modelo en Drive: fotoperitacion/damage_seg/phase2_finetune/weights/best.pt")

## 7. (Opcional) Modelo de partes — localización por zona
Entrena el modelo `carparts-seg` para asignar cada daño a una zona (front/rear/FL/FR/RL/RR).

In [ ]:
!python scripts/train_parts.py --batch 16   # carparts-seg se descarga solo
!cp -r runs/parts_seg "$WORK/"
print("✅ Partes en Drive: fotoperitacion/parts_seg/train/weights/best.pt")